In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from datetime import datetime

# Widget with default value
dbutils.widgets.text("table_name", "dim_sellers")
table_name = dbutils.widgets.get("table_name")

# Configuration & Paths
silver_base = "abfss://processed@[YOUR_ACCOUNT_STORAGE_NAME].dfs.core.windows.net/silver/"
gold_base = "abfss://curated@[YOUR_ACCOUNT_STORAGE_NAME].dfs.core.windows.net/gold/"
target_table = table_name
gold_path = f"{gold_base}{target_table}"

# Silver Sources
df_sellers = spark.read.format("delta").load(f"{silver_base}olist_sellers_dataset")
df_geo = spark.read.format("delta").load(f"{silver_base}olist_geolocation_dataset")

# Transformation logic
df_dim_base = (df_sellers.alias("s")
               .join(df_geo.alias("g"), F.col("s.seller_zip_code_prefix") == F.col("g.geolocation_zip_code_prefix"), "left")
               .select(
                   F.md5(F.concat(F.col("s.seller_id"), F.col("s.start_date").cast("string"))).alias("seller_key"),
                   "s.seller_id",
                   "s.seller_city",
                   "s.seller_state",
                   "s.seller_zip_code_prefix",
                   F.col("g.geolocation_lat").alias("latitude"),
                   F.col("g.geolocation_lng").alias("longitude"),
                   "s.start_date",
                   "s.end_date",
                   "s.is_current",
                   F.date_trunc("second", F.current_timestamp()).alias("gold_load_at")
               ))

# Unknown record
now = datetime.now().replace(microsecond=0)
unknown_data = [("-1", "UNKNOWN", "NA", "NA", None, None, None, datetime(1900, 1, 1), None, True, now)]
target_schema = df_dim_base.schema
df_unknown = spark.createDataFrame(unknown_data, schema=target_schema)
df_final = df_dim_base.unionByName(df_unknown)

# Incremental Merge (Upsert)
print(f"--> Starting Gold Load: {target_table}")

if DeltaTable.isDeltaTable(spark, gold_path):
    dt_gold = DeltaTable.forPath(spark, gold_path)

    # Merge Condition
    data_cols = ["seller_city", "seller_state", "latitude", "longitude", "is_current"]
    change_condition = " OR ".join([f"NOT (target.{c} <=> source.{c})" for c in data_cols])

    (dt_gold.alias("target")
     .merge(df_final.alias("source"), "target.seller_key = source.seller_key")
     .whenMatchedUpdateAll(condition = change_condition)
     .whenNotMatchedInsertAll()
     .execute())
    
    print(f"--> [Merge] {target_table} updated.")
else:
    #First time load
    df_final.write.format("delta").mode("overwrite").save(gold_path)
    print(f"--> [INIT] {target_table} created.")

# Audit & Exit
dt_final = DeltaTable.forPath(spark, gold_path)

last_operation = dt_final.history(1).select("operationMetrics").collect()[0][0]
rows_inserted = int(last_operation.get("numTargetRowsInserted", 0))
rows_updated = int(last_operation.get("numTargetRowsUpdated", 0))
total_rows = last_operation.get("numOutputRows", "Check History")

print("-" * 50)
print(f"--> Table: {table_name} | Status: Success")
print(f"--> Rows Processed in last run: {rows_inserted + rows_updated}")
print("-" * 50)

dbutils.notebook.exit(f"Success | Inserted: {rows_inserted} | Updated: {rows_updated}")